# Full Online Evaluation — Baseline Variant

Runs the complete cascade (Global Detector → EWMA smoothing → Specialist committee → TOPSIS ranking) over the full test set (8.64M samples, 18 fault classes) and reports the per-class and macro-averaged metrics (BACC, FDR, FAR, F1, G-Mean, Accuracy, Kappa) used in the thesis results chapter.

In [ ]:
import os, joblib, warnings
import numpy as np
import pandas as pd
import pyreadr
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score

# Silencia avisos técnicos do XGBoost
warnings.filterwarnings('ignore')

# ==============================================================================
# NOTE: paths below are relative to this notebook living in /notebooks.
# Place the TEP dataset files in /data and trained model artifacts in /models
# (see the repository README for download links and folder layout).
# 0. CONFIGURAÇÕES
# ==============================================================================
PASTA_MODELOS = os.path.join("..", "models", "baseline")
CAMINHO_FALHAS = os.path.join("..", "data", "TEP_Faulty_Testing.RData")
CAMINHO_NORMAL_TESTE = os.path.join("..", "data", "TEP_FaultFree_Testing.RData") 
FALHAS_EXCLUIDAS = [3, 9, 15] 

# ==============================================================================
# 1. CARREGAMENTO LIMPO (RUN COMPLETO 0–960)
# ==============================================================================
print("⏳ Carregando bases de teste...")

res_f = pyreadr.read_r(CAMINHO_FALHAS)
df_todas = res_f[list(res_f.keys())[0]]

res_n = pyreadr.read_r(CAMINHO_NORMAL_TESTE)
df_normal_bruto = res_n[list(res_n.keys())[0]]

df_teste = pd.DataFrame()
rng = np.random.RandomState(99)
RUNS_POR_CLASSE = 500  

# ------------------------------------------------------------------------------
# 1.1 COLETANDO AS FALHAS E CLASSE NORMAL
# ------------------------------------------------------------------------------
for f in range(1, 21):
    if f in FALHAS_EXCLUIDAS:
        continue
    df_f_base = df_todas[df_todas['faultNumber'] == f].copy()
    runs_f = df_f_base['simulationRun'].unique()
    runs_esc = rng.choice(runs_f, size=RUNS_POR_CLASSE, replace=False)
    df_f_sample = df_f_base[df_f_base['simulationRun'].isin(runs_esc)].copy()
    df_teste = pd.concat([df_teste, df_f_sample])

runs_norm = df_normal_bruto['simulationRun'].unique()
runs_n_esc = rng.choice(runs_norm, size=RUNS_POR_CLASSE, replace=False)
df_norm_sample = df_normal_bruto[df_normal_bruto['simulationRun'].isin(runs_n_esc)].copy()
df_norm_sample['faultNumber'] = 0
if 'target' in df_norm_sample.columns:
    df_norm_sample['target'] = 0

df_teste = pd.concat([df_teste, df_norm_sample])

df_teste = df_teste.sort_values(
    by=['faultNumber', 'simulationRun', 'sample']
).reset_index(drop=True)

print(f"✅ Dataset montado: {len(df_teste)} amostras.")


# ==============================================================================
# 2. CARREGAMENTO DOS MODELOS
# ==============================================================================
pacote_global = joblib.load(os.path.join(PASTA_MODELOS, "detector_global.joblib"))
modelo_global = pacote_global['modelo']
features_globais = pacote_global['features']

ALPHA = 0.20 
THRESHOLD_PORTEIRO = 0.35 

comite = []
for arquivo in os.listdir(PASTA_MODELOS):
    if arquivo.startswith("especialista_f") and arquivo.endswith(".joblib"):
        pacote = joblib.load(os.path.join(PASTA_MODELOS, arquivo))
        if pacote['falha_id'] not in FALHAS_EXCLUIDAS:
            comite.append(pacote)


# ==============================================================================
# 3. SIMULAÇÃO OTIMIZADA EM LOTE
# ==============================================================================
import time
start_time = time.time()
print(f"\n🚀 Iniciando diagnóstico VETORIZADO...")

y_true = np.where(df_teste['sample'] <= 160, 0, df_teste['faultNumber'].values.astype(int))
y_pred = np.zeros(len(df_teste), dtype=int)

df_teste['prob_bruta'] = modelo_global.predict_proba(df_teste[features_globais])[:, 1]
df_teste['prob_suave'] = df_teste.groupby(['faultNumber', 'simulationRun'])['prob_bruta'] \
                                 .ewm(alpha=ALPHA, adjust=False).mean().values

mascara_anomalias = df_teste['prob_suave'] >= THRESHOLD_PORTEIRO
indices_anomalias = df_teste[mascara_anomalias].index

if len(indices_anomalias) > 0:
    df_anomalias = df_teste.loc[indices_anomalias]
    preds_especialistas = {}
    for esp in comite:
        preds_especialistas[esp['falha_id']] = esp['modelo'].predict_proba(df_anomalias[esp['features']])[:, 1]
    
    pesos = np.array([0.60, 0.10, 0.10, 0.05, 0.15])
    N = len(indices_anomalias) 
    E = len(comite)            
    matriz_topsis = np.zeros((N, E, 5)) 
    falhas_ids = np.zeros(E, dtype=int)
    
    for idx_e, esp in enumerate(comite):
        falhas_ids[idx_e] = esp['falha_id']
        matriz_topsis[:, idx_e, 0] = preds_especialistas[esp['falha_id']] 
        matriz_topsis[:, idx_e, 1] = esp['acuracia']                      
        matriz_topsis[:, idx_e, 2] = esp.get('precisao', 0.95)            
        matriz_topsis[:, idx_e, 3] = esp.get('recall', 0.95)              
        matriz_topsis[:, idx_e, 4] = esp.get('f1', 0.95)                  
        
    norma = np.linalg.norm(matriz_topsis, axis=1, keepdims=True) + 1e-10
    matriz_norm = (matriz_topsis / norma) * pesos
    
    id_pos = np.max(matriz_norm, axis=1, keepdims=True)
    id_neg = np.min(matriz_norm, axis=1, keepdims=True)
    
    dist_pos = np.sqrt(np.sum((matriz_norm - id_pos)**2, axis=2))
    dist_neg = np.sqrt(np.sum((matriz_norm - id_neg)**2, axis=2))
    
    score_topsis = dist_neg / (dist_pos + dist_neg + 1e-10)
    vencedor_idx = np.argmax(score_topsis, axis=1)
    y_pred[indices_anomalias] = falhas_ids[vencedor_idx]


# ==============================================================================
# 4. RESULTADO FINAL E MÉTRICAS CIENTÍFICAS DE DIAGNÓSTICO
# ==============================================================================
kappa = cohen_kappa_score(y_true, y_pred)
print("\n" + "="*86)
print(f"⭐ DESEMPENHO GLOBAL DO FRAMEWORK XFDDC")
print(f"👉 ACURÁCIA GLOBAL SIMPLES (ACC) : {accuracy_score(y_true, y_pred)*100:.2f}%")
print(f"🎯 ÍNDICE KAPPA (Concordância)   : {kappa:.4f}")
print("="*86)

# Cabeçalho estruturado com espaçamentos fixos
print(f"{'Classe de Falha':<18} | {'BACC':<10} | {'FDR (Recall)':<14} | {'FAR':<10} | {'F1-Score':<10} | {'G-Mean':<10}")
print("-" * 86)

metricas_acumuladas = []
falhas_presentes = sorted(np.unique(y_true))

for f in falhas_presentes:
    TP = np.sum((y_true == f) & (y_pred == f))  
    FN = np.sum((y_true == f) & (y_pred != f))  
    FP = np.sum((y_true != f) & (y_pred == f))  
    TN = np.sum((y_true != f) & (y_pred != f))  
    
    FDR = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    FAR = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    Spec = TN / (FP + TN) if (FP + TN) > 0 else 0.0
    Precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    
    BACC = (FDR + Spec) / 2.0
    G_Mean = np.sqrt(FDR * Spec)
    F1 = (2 * Precision * FDR) / (Precision + FDR) if (Precision + FDR) > 0 else 0.0
    
    metricas_acumuladas.append([BACC, FDR, FAR, F1, G_Mean])
    
    nome_falha = "Normal (00)" if f == 0 else f"Falha {f:02d}"
    
    # Print síncrono com espaçamento rígido à esquerda e à direita
    print(f"{nome_falha:<18} | {BACC*100:<8.2f}% | {FDR*100:<12.2f}% | {FAR*100:<8.2f}% | {F1*100:<8.2f}% | {G_Mean*100:<8.2f}%")

print("-" * 86)

# Processamento matemático da Média Macro Global
matriz_resumos = np.array(metricas_acumuladas)
macro_averages = np.mean(matriz_resumos, axis=0)

# Linha de encerramento acadêmico
print(f"{'GLOBAL (Macro Média)':<18} | {macro_averages[0]*100:<8.2f}% | {macro_averages[1]*100:<12.2f}% | {macro_averages[2]*100:<8.2f}% | {macro_averages[3]*100:<8.2f}% | {macro_averages[4]*100:<8.2f}%")
print("="*86)

print("\n📝 Relatório Clássico (Sklearn):")
print(classification_report(y_true, y_pred, zero_division=0))